# 导入依赖和文件

In [9]:
import json
import joblib
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.metrics import accuracy_score, classification_report,log_loss
from sklearn.ensemble import RandomForestClassifier

In [2]:

# 加载特征数据
X = np.load("X.npy")
X_test = np.load("X_test.npy")
y = np.load("y.npy")

# 划分训练集与验证集

X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, train_size=0.8, test_size=0.2, random_state=0)


# 寻找并保存最优参数

In [ ]:
对于基学习器，选用logloss作为指标

In [ ]:
def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.1),
        'random_state': 0,
        'verbose': -1
    }
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_valid)[:, 1]
    return log_loss(y_valid, y_pred_proba)

study = optuna.create_study(direction='minimize')
study.optimize(lgbm_objective, n_trials=200)
print(f"Best LGBM parameters: {study.best_params}")
print(f"Best logloss value: {study.best_value}")

# 创建 Optuna 研究对象并进行优化

lgbm_study = optuna.create_study(direction='minimize')
lgbm_study.optimize(lgbm_objective, n_trials=300)



[I 2025-04-27 17:42:50,801] A new study created in memory with name: no-name-261b3681-9765-458c-8398-0a66ac20e88f
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 17:42:51,118] Trial 0 finished with value: 0.41824320742204474 and parameters: {'n_estimators': 450, 'learning_rate': 0.03926703843828645, 'max_depth': 6, 'num_leaves': 164, 'subsample': 0.9863254405784865, 'colsample_bytree': 0.9633257689980763, 'reg_alpha': 0.7707308815332601, 'reg_lambda': 0.4392400029413208, 'min_split_gain': 0.02938391560284294}. Best is trial 0 with value: 0.41824320742204474.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 20

Best LGBM parameters: {'n_estimators': 982, 'learning_rate': 0.02224232023027439, 'max_depth': 4, 'num_leaves': 157, 'subsample': 0.7554381760171093, 'colsample_bytree': 0.9927272290883854, 'reg_alpha': 0.17927926980645065, 'reg_lambda': 0.1344995261420041, 'min_split_gain': 0.0018985868007980488}
Best logloss value: 0.4046426082051519


/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 17:44:17,042] Trial 0 finished with value: 0.49486502233998564 and parameters: {'n_estimators': 636, 'learning_rate': 0.0015069958705865622, 'max_depth': 7, 'num_leaves': 60, 'subsample': 0.7512567951104099, 'colsample_bytree': 0.9074417689609449, 'reg_alpha': 0.01498000569921365, 'reg_lambda': 0.3509428013261655, 'min_split_gain': 0.06002457387794918}. Best is trial 0 with value: 0.49486502233998564.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 17:44:17,415] Trial 1 finished with value: 0.42750996821279735 and parameters: {'n_estimators': 473, 'l

KeyboardInterrupt: 

In [19]:
def catboost_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'depth': trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'border_count': trial.suggest_int('border_count', 50, 255),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 30),
        'rsm': trial.suggest_float('rsm', 0.7, 1.0),
        'verbose': False,
        'random_state': 0
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_valid)[:, 1]
    return log_loss(y_valid, y_pred_proba)



# 创建 Optuna 研究对象并进行优化

catboost_study = optuna.create_study(direction='minimize')
catboost_study.optimize(catboost_objective, n_trials=300)



[I 2025-04-27 17:53:12,901] A new study created in memory with name: no-name-ed59a756-5816-4530-a95d-d0b99c503ee9
[I 2025-04-27 17:53:14,021] Trial 0 finished with value: 0.4495033030527878 and parameters: {'n_estimators': 818, 'learning_rate': 0.2684511369369536, 'depth': 4, 'l2_leaf_reg': 8.607765282654896, 'border_count': 90, 'min_data_in_leaf': 29, 'rsm': 0.8223777123471422}. Best is trial 0 with value: 0.4495033030527878.
[I 2025-04-27 17:53:19,306] Trial 1 finished with value: 0.4720580118898347 and parameters: {'n_estimators': 878, 'learning_rate': 0.07580836162544177, 'depth': 9, 'l2_leaf_reg': 4.257003836669768, 'border_count': 199, 'min_data_in_leaf': 17, 'rsm': 0.7273598629836713}. Best is trial 0 with value: 0.4495033030527878.
[I 2025-04-27 17:53:20,809] Trial 2 finished with value: 0.40584967361459623 and parameters: {'n_estimators': 1060, 'learning_rate': 0.06016106682178436, 'depth': 4, 'l2_leaf_reg': 1.0588068283155885, 'border_count': 193, 'min_data_in_leaf': 22, 'rsm

In [20]:
# 获取最优参数
lgbm_best_params = lgbm_study.best_params
lgbm_best_value = lgbm_study.best_value
catboost_best_params = catboost_study.best_params
catboost_best_value = catboost_study.best_value

In [21]:
print("LGBM 最佳得分:", lgbm_best_value)
print("CatBoost 最佳得分:", catboost_best_value)
# 导出 LGBM 和 CatBoost 的最优参数到 JSON 文件
with open('lgbm_best_params.json', 'w') as f:
    json.dump(lgbm_best_params, f)

with open('catboost_best_params.json', 'w') as f:
    json.dump(catboost_best_params, f)

LGBM 最佳得分: 0.4018925052043311
CatBoost 最佳得分: 0.4017258255034015


# 训练并保存模型

In [22]:

# 导入最优参数
with open('lgbm_best_params.json', 'r') as f:
    lgbm_best_params = json.load(f)

with open('catboost_best_params.json', 'r') as f:
    catboost_best_params = json.load(f)

# 使用最优参数重新实例化模型并训练
lgbm_model = LGBMClassifier(**lgbm_best_params, random_state=0)
catboost_model = CatBoostClassifier(**catboost_best_params, verbose=False, random_state=0)

lgbm_model.fit(X_train, y_train)
catboost_model.fit(X_train, y_train)

joblib.dump(lgbm_model, 'lgbm_best_model.joblib')
joblib.dump(catboost_model, 'catboost_best_model.joblib')


['catboost_best_model.joblib']

In [23]:
# 加载模型
lgbm_loaded_model = joblib.load('lgbm_best_model.joblib')
catboost_loaded_model = joblib.load('catboost_best_model.joblib')

# 对测试数据进行预测，获取概率值
lgbm_prob = lgbm_model.predict_proba(X_test)[:, 1]
catboost_prob = catboost_model.predict_proba(X_test)[:, 1]

# 简单平均融合
ensemble_prob = (lgbm_prob + catboost_prob) / 2

# 根据阈值生成最终预测
threshold = 0.5
ensemble_pred = ensemble_prob > threshold

# 加载测试数据
test_data = pd.read_csv('test.csv')  # 确保文件路径正确

# 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],  # 确保这里使用正确的列名
    'Transported': ensemble_pred
})

# 保存为 CSV 文件
submission.to_csv('submission.csv', index=False)


/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# 模型融合与提交文件

基学习器已经训练好，现在使用元学习器进行模型融合

In [ ]:
# # 加载训练好的基模型
# lgbm_model = joblib.load('lgbm_best_model.joblib')
# catboost_model = joblib.load('catboost_best_model.joblib')

# # 检查模型加载是否正确
# print(lgbm_model.get_params())
# print(catboost_model.get_params())

# # 定义生成元特征的函数
# def generate_meta_features(model, X_train, y_train, X_valid, y_valid, X_test, n_splits=5):
#     print("Entering generate_meta_features function")  # 调试信息
#     kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
#     meta_train = np.zeros((X_train.shape[0],))
#     meta_valid = np.zeros((X_valid.shape[0],))
#     meta_test = np.zeros((X_test.shape[0],))
    
#     for train_index, val_index in kf.split(X_train):
#         print("Inside KFold loop")  # 调试信息
#         X_tr, X_val = X_train[train_index], X_train[val_index]
#         y_tr = y_train[train_index]  # 获取当前折的训练目标变量
        
#         # 确保 X_tr 和 y_tr 的维度正确
#         print(f"X_tr shape: {X_tr.shape}, y_tr shape: {y_tr.shape}")
        
#         model.fit(X_tr, y_tr)  # 传递 y_tr 作为目标变量
#         meta_train[val_index] = model.predict_proba(X_val)[:, 1]
#         print("Completed a fold")  # 调试信息
    
#     # 使用整个训练集重新训练模型以生成验证集和测试集的元特征
#     print("Refitting model on entire training set")  # 调试信息
#     model.fit(X_train, y_train)
#     meta_valid = model.predict_proba(X_valid)[:, 1]
#     meta_test = model.predict_proba(X_test)[:, 1]
#     print("Exiting generate_meta_features function")  # 调试信息
    
#     return meta_train, meta_valid, meta_test

# # 为 LGBM 和 CatBoost 生成元特征
# print("Generating meta features for LGBM")  # 调试信息
# lgbm_meta_train, lgbm_meta_valid, lgbm_meta_test = generate_meta_features(lgbm_model, X_train, y_train, X_valid, y_valid, X_test)
# print("Generating meta features for CatBoost")  # 调试信息
# catboost_meta_train, catboost_meta_valid, catboost_meta_test = generate_meta_features(catboost_model, X_train, y_train, X_valid, y_valid, X_test)

# # 构建元特征矩阵
# X_train_meta = np.column_stack((lgbm_meta_train, catboost_meta_train))
# X_valid_meta = np.column_stack((lgbm_meta_valid, catboost_meta_valid))
# X_test_meta = np.column_stack((lgbm_meta_test, catboost_meta_test))

# # 定义元模型的目标函数
# def meta_objective(trial):
#     params = {
#         'C': trial.suggest_float('C', 0.01, 10.0, log=True),
#         'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']),
#         'random_state': 0
#     }
#     model = LogisticRegression(**params)
#     model.fit(X_train_meta, y_train)
#     y_pred = model.predict(X_valid_meta)
#     accuracy = accuracy_score(y_valid, y_pred)
#     print(f"Meta model accuracy: {accuracy:.4f}")  # 调试信息
#     return accuracy

# # 创建 Optuna 研究对象并进行优化
# print("Starting Optuna study for meta model")  # 调试信息
# meta_study = optuna.create_study(direction='maximize')
# meta_study.optimize(meta_objective, n_trials=50)

# # 获取最优参数
# meta_best_params = meta_study.best_params
# meta_best_value = meta_study.best_value

# print(f"元模型最优参数: {meta_best_params}")
# print(f"元模型最优准确率: {meta_best_value:.4f}")

# # 使用最优参数训练最终的元模型
# best_meta_model = LogisticRegression(**meta_best_params, random_state=0)
# best_meta_model.fit(X_train_meta, y_train)

# # 保存元模型
# joblib.dump(best_meta_model, 'best_meta_model.joblib')


In [26]:
# 加载训练好的基模型
lgbm_model = joblib.load('lgbm_best_model.joblib')
catboost_model = joblib.load('catboost_best_model.joblib')

# 检查模型加载是否正确
print("LGBM Model Parameters:", lgbm_model.get_params())
print("CatBoost Model Parameters:", catboost_model.get_params())

# 定义生成元特征的函数
def generate_meta_features(model, X_train, y_train, X_valid, y_valid, X_test, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    meta_train = np.zeros((X_train.shape[0],))
    meta_valid = np.zeros((X_valid.shape[0],))
    meta_test = np.zeros((X_test.shape[0],))
    
    for train_index, val_index in kf.split(X_train):
        X_tr, X_val = X_train[train_index], X_train[val_index]
        y_tr = y_train[train_index]
        model.fit(X_tr, y_tr)
        # 使用预测概率作为元特征
        meta_train[val_index] = model.predict_proba(X_val)[:, 1]
    
    # 使用整个训练集重新训练模型以生成验证集和测试集的元特征
    model.fit(X_train, y_train)
    # 使用预测概率作为元特征
    meta_valid = model.predict_proba(X_valid)[:, 1]
    meta_test = model.predict_proba(X_test)[:, 1]
    
    return meta_train, meta_valid, meta_test


print("Generating meta features for LGBM")
lgbm_meta_train, lgbm_meta_valid, lgbm_meta_test = generate_meta_features(lgbm_model, X_train, y_train, X_valid, y_valid, X_test)

print("Generating meta features for CatBoost")
catboost_meta_train, catboost_meta_valid, catboost_meta_test = generate_meta_features(catboost_model, X_train, y_train, X_valid, y_valid, X_test)

# 构建元特征矩阵
X_train_meta = np.column_stack((lgbm_meta_train, catboost_meta_train))
X_valid_meta = np.column_stack((lgbm_meta_valid, catboost_meta_valid))
X_test_meta = np.column_stack((lgbm_meta_test, catboost_meta_test))

# 定义元学习器的超参数优化目标函数
def logistic_regression_objective(trial):
    params = {
        'C': trial.suggest_float('C', 0.01, 10.0, log=True),
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'sag', 'saga']),
        'random_state': 0
    }
    model = LogisticRegression(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    return accuracy_score(y_valid, y_pred)

def random_forest_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'random_state': 0
    }
    model = RandomForestClassifier(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    return accuracy_score(y_valid, y_pred)

def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': trial.suggest_int('num_leaves', 10, 200),
        'random_state': 0
    }
    model = LGBMClassifier(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    return accuracy_score(y_valid, y_pred)

def catboost_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 5, 16),
        'random_state': 0
    }
    model = CatBoostClassifier(**params, verbose=0)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    return accuracy_score(y_valid, y_pred)

def lasso_objective(trial):
    params = {
        'alpha': trial.suggest_float('alpha', 0.0001, 10.0, log=True),
        'random_state': 0
    }
    model = Lasso(**params)
    model.fit(X_train_meta, y_train)
    y_pred = model.predict(X_valid_meta)
    y_pred_class = (y_pred > 0.5).astype(int)
    return accuracy_score(y_valid, y_pred_class)

# 创建Optuna研究对象并进行优化
lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(lgbm_objective, n_trials=100)

catboost_study = optuna.create_study(direction='maximize')
catboost_study.optimize(catboost_objective, n_trials=100)

logistic_regression_study = optuna.create_study(direction='maximize')
logistic_regression_study.optimize(logistic_regression_objective, n_trials=100)

random_forest_study = optuna.create_study(direction='maximize')
random_forest_study.optimize(random_forest_objective, n_trials=100)

lasso_study = optuna.create_study(direction='maximize')
lasso_study.optimize(lasso_objective, n_trials=100)

# 获取最优参数和准确率
meta_models = {
    'LogisticRegression': {
        'best_params': logistic_regression_study.best_params,
        'best_accuracy': logistic_regression_study.best_value
    },
    'RandomForest': {
        'best_params': random_forest_study.best_params,
        'best_accuracy': random_forest_study.best_value
    },
    'LGBMClassifier': {
        'best_params': lgbm_study.best_params,
        'best_accuracy': lgbm_study.best_value
    },
    'CatBoostClassifier': {
        'best_params': catboost_study.best_params,
        'best_accuracy': catboost_study.best_value
    },
    'LassoRegression': {
        'best_params': lasso_study.best_params,
        'best_accuracy': lasso_study.best_value
    }
}

# 选择最佳元学习器
best_meta_model_name = max(meta_models, key=lambda k: meta_models[k]['best_accuracy'])
best_meta_model_params = meta_models[best_meta_model_name]['best_params']
best_meta_model_accuracy = meta_models[best_meta_model_name]['best_accuracy']


# 根据最佳元学习器的名称创建模型实例并训练
if best_meta_model_name == 'LogisticRegression':
    best_meta_model = LogisticRegression(**best_meta_model_params, random_state=0)
elif best_meta_model_name == 'RandomForest':
    best_meta_model = RandomForestClassifier(**best_meta_model_params, random_state=0)
elif best_meta_model_name == 'LGBMClassifier':
    best_meta_model = LGBMClassifier(**best_meta_model_params, random_state=0)
elif best_meta_model_name == 'CatBoostClassifier':
    best_meta_model = CatBoostClassifier(**best_meta_model_params, random_state=0, verbose=0)
elif best_meta_model_name == 'LassoRegression':
    best_meta_model = Lasso(**best_meta_model_params, random_state=0)

# 如果是Lasso回归，需要将预测结果转换为分类标签
if best_meta_model_name == 'LassoRegression':
    best_meta_model.fit(X_train_meta, y_train)
    y_pred_final = best_meta_model.predict(X_valid_meta)
    y_pred_final_class = (y_pred_final > 0.5).astype(int)
    final_accuracy = accuracy_score(y_valid, y_pred_final_class)
    final_report = classification_report(y_valid, y_pred_final_class)
else:
    best_meta_model.fit(X_train_meta, y_train)
    y_pred_final = best_meta_model.predict(X_valid_meta)
    final_accuracy = accuracy_score(y_valid, y_pred_final)
    final_report = classification_report(y_valid, y_pred_final)

# 保存元模型
joblib.dump(best_meta_model, 'best_meta_model.joblib')


LGBM Model Parameters: {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.9207371638767929, 'importance_type': 'split', 'learning_rate': 0.18652943744318107, 'max_depth': 5, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.004148196536996891, 'n_estimators': 129, 'n_jobs': None, 'num_leaves': 10, 'objective': None, 'random_state': 0, 'reg_alpha': 0.0002383668653049495, 'reg_lambda': 0.01545250615707709, 'subsample': 0.8696301542453486, 'subsample_for_bin': 200000, 'subsample_freq': 0}
CatBoost Model Parameters: {'learning_rate': 0.018896259609261473, 'depth': 6, 'l2_leaf_reg': 4.553196577575439, 'rsm': 0.8979677144701962, 'border_count': 246, 'verbose': False, 'n_estimators': 1354, 'random_state': 0, 'min_data_in_leaf': 19}
Generating meta features for LGBM


/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site

Generating meta features for CatBoost


[I 2025-04-27 18:13:31,619] A new study created in memory with name: no-name-6441bc2a-b9f4-4ad6-bc23-f2bff33656b8
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 18:13:32,090] Trial 0 finished with value: 0.7469810235767682 and parameters: {'n_estimators': 801, 'learning_rate': 0.2400254544846589, 'max_depth': 20, 'num_leaves': 169}. Best is trial 0 with value: 0.7469810235767682.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-27 18:13:32,221] Trial 1 finished with value: 0.7607820586543991 and parameters: {'n_estimators': 739, 'learning_rate': 0.2843392344826799, 'max_depth': 5, 'num_leaves': 104}.

['best_meta_model.joblib']

In [31]:
# 保存最佳参数到JSON文件
best_params = {
    'LogisticRegression': logistic_regression_study.best_params,
    'RandomForest': random_forest_study.best_params,
    'LGBMClassifier': lgbm_study.best_params,
    'CatBoostClassifier': catboost_study.best_params,
    'LassoRegression': lasso_study.best_params
}

with open('best_meta_params.json', 'w') as f:
    json.dump(best_params, f, indent=4)

In [32]:
for name, metrics in meta_models.items():
    print(f"{name}: Best Accuracy = {metrics['best_accuracy']:.4f}, Best Params = {metrics['best_params']}")
print(f"\n{best_meta_model} Evaluation:")
print(f"Accuracy: {final_accuracy:.4f}")
print(f"Classification Report:\n{final_report}")

LogisticRegression: Best Accuracy = 0.8160, Best Params = {'C': 0.05252477605657189, 'solver': 'liblinear'}
RandomForest: Best Accuracy = 0.8102, Best Params = {'n_estimators': 938, 'max_depth': 17, 'min_samples_split': 15, 'min_samples_leaf': 8}
LGBMClassifier: Best Accuracy = 0.8039, Best Params = {'n_estimators': 141, 'learning_rate': 0.03375583072352935, 'max_depth': 14, 'num_leaves': 59}
CatBoostClassifier: Best Accuracy = 0.8125, Best Params = {'n_estimators': 187, 'learning_rate': 0.05245667866027533, 'max_depth': 5}
LassoRegression: Best Accuracy = 0.8137, Best Params = {'alpha': 0.11216090925269158}

LogisticRegression(C=0.05252477605657189, random_state=0, solver='liblinear') Evaluation:
Accuracy: 0.8160
Classification Report:
              precision    recall  f1-score   support

       False       0.83      0.79      0.81       863
        True       0.80      0.84      0.82       876

    accuracy                           0.82      1739
   macro avg       0.82      0.82  

In [33]:
# 加载元模型
best_meta_model = joblib.load('best_meta_model.joblib')

# 生成预测结果
pred = best_meta_model.predict(X_test_meta)

# 加载测试数据
test_data = pd.read_csv('test.csv')

# 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Transported': pred > 0.5
})

# 保存为 CSV 文件
submission.to_csv('submission.csv', index=False)